In [43]:
# --- 1. Configuration Section ---
Month_ = 2
Year_ = 2569
User_ = ["All"]
sessions = 'g' #กำหนดชุด g ทั่วไป, u นอกเขต

# --- Section1: Input Database ---
realTimeDataformGoogleSheet = 0

# --- Section2 : Data Analysis ---
web = 0             # โหลดข้อมูลจากแหล่งSSO-TPSOตรวจสอบสินค้า
standard_price = 0  # โหลดราคากลางทุกชุด
shopBKK = 1         # โหลดข้อมูลราคากรุงเทพ
TimeSerie = 0     # โหลดข้อมูลเวลาเทียบความผันผวน
detect = 0          # ตรวจสอบสินค้าเทียบ z-score
excess_lost = 0    # นับจำนวนขาดเกิน

# --- Section3: Final Output & Upload ---
upload = 0

### Condition Input

In [44]:
KERNEL_NAME = 'python3'
if sessions == 'g':
    web_cpiG = web
    detectG = detect
    time_serieG = TimeSerie
    excess_lostG = excess_lost      
elif sessions == 'u':
    web_cpiU = web
    detectU = detect
    time_serieU = TimeSerie
    excess_lostU = excess_lost
else:
    error

In [45]:
import os, sys, glob, io, contextlib, re, time
import papermill as pm
import pandas as pd
import nbformat
from tqdm.notebook import tqdm
from IPython.display import display


In [46]:
# pip install --upgrade jupyter ipywidgets pip install papermill
# !pip install ipywidgets

###  Helper Functions

In [47]:
# --- 2.  ---
def get_nb_progress(nb_path):
    try:
        with open(nb_path, 'r', encoding='utf-8') as f:
            nb = nbformat.read(f, as_version=4)
        code_cells = [c for c in nb.cells if c.cell_type == 'code']
        total = len(code_cells)
        executed = len([c for c in code_cells if c.get('execution_count') is not None])
        return round((executed / total) * 100, 1) if total > 0 else 0.0
    except Exception:
        return 0.0

def run_notebooks(input_path, specific_files=None, **params):
    current_script = os.path.basename(sys.argv[0])
    folders = input_path if isinstance(input_path, list) else [input_path]
    
    tasks = []
    for folder in folders:
        if not os.path.exists(folder): continue
        abs_path = os.path.abspath(folder)
        if specific_files:
            files = [os.path.join(abs_path, f) for f in (specific_files if isinstance(specific_files, list) else [specific_files])]
        else:
            files = sorted([f for f in glob.glob(os.path.join(abs_path, "*.ipynb")) 
                           if os.path.basename(f) != current_script and ".ipynb_checkpoints" not in f])
        for f in files: tasks.append((f, abs_path))

    if not tasks: return []

    results = []
    # แก้ไข custom_format: ใช้ {desc} แทน {postfix} และลบลูกน้ำในนี้ออกให้หมด
    # เราจะไปคุมหน้าตาของ [00:03 , ไฟล์ = ...] ในตัวแปร desc แทน
    custom_format = '{percentage:3.0f}%|{bar}| [{elapsed}{desc}]'    

    with tqdm(total=100, bar_format=custom_format, leave=True) as pbar:
        for nb_path, abs_path in tasks:
            name_no_ext = os.path.splitext(os.path.basename(nb_path))[0]
            
            # แก้ไขตรงนี้: ใช้ set_description เพื่อส่ง String ที่เราจัดรูปแบบเอง
            # ใส่ " , ไฟล์ =" เข้าไปดื้อๆ เลย จะไม่มีลูกน้ำจากระบบมาแทรกแน่นอน
            pbar.set_description(f" , ไฟล์ = {name_no_ext}, ตัวแปร={params}")
            
            pbar.reset(total=100) 
            start_time = time.time()
            success = False
            
            f_err = io.StringIO()
            with contextlib.redirect_stderr(f_err):
                try:
                    pm.execute_notebook(
                        nb_path, nb_path, 
                        kernel_name=KERNEL_NAME,
                        parameters=params,
                        log_output=False,
                        progress_bar=False, 
                        cwd=abs_path 
                    )
                    success = True
                    pbar.n = 100 
                except Exception:
                    pbar.n = get_nb_progress(nb_path) 

            pbar.refresh() 
            duration = time.time() - start_time
            
            results.append({
                "ชื่อไฟล์": name_no_ext,
                "ตัวแปรที่ใช้": str(params) if params else "-",
                "เวลา (วินาที)": round(duration, 2),
                "รันไปได้ (%)": pbar.n,
                "สถานะ": "✅" if success else "❌"
            })
    return results

### Main Execution Section

In [48]:
# --- 3. Main Execution Section ---
if __name__ == "__main__":
    all_reports = []
    start_all = time.time()

# --- Section1: Input Database ---
    if realTimeDataformGoogleSheet  == 1:
        print("\n📥 1 Input: โหลดข้อมูลตั้งต้น Master/Supplementary/ราคากลาง")
        all_reports.extend(run_notebooks("01-database", specific_files=["Master.ipynb"]))
        all_reports.extend(run_notebooks("01-database", specific_files=["Supplementaryfilre.ipynb"]))
        all_reports.extend(run_notebooks("01-database", specific_files=["StandardPriceAutomation.ipynb"], update_sort_set=False, download_set=True))

    # --- Section2.1: Data Analysis (Session G) ---
    if sessions == "g":
        print("\n📊 2 Data Analysis: คำนวณชุดทั่วไป (G)")
        if shopBKK == 1:
            all_reports.extend(run_notebooks("01-database", specific_files=["ShopBKK.ipynb"]))
        
        if web_cpiG == 1:
            all_reports.extend(run_notebooks("01-database", specific_files=["Data_CPI_G.ipynb"], Month=Month_, Year=Year_))
            # all_reports.extend(run_notebooks("01-database", specific_files=["data01_cpig_month.ipynb"], Month=Month_, Year=Year_))
            # all_reports.extend(run_notebooks("01-database", specific_files=["data02_cpig_week.ipynb"], Month=Month_, Year=Year_))
        
        if time_serieG == 1:
            all_reports.extend(run_notebooks("02-timeseries_data", specific_files=["Timeseries_ชุดทั่วไป_เดือน.ipynb"], Month=Month_, Year=Year_))
        
        if excess_lostG == 1:
            all_reports.extend(run_notebooks("04-Excess_Lost", specific_files=["Excess_Lost.ipynb"], Month=Month_, Year=Year_, session='g'))
        
        if standard_price == 1:
            all_reports.extend(run_notebooks("01-database", specific_files=["ราคากลาง_concat.ipynb"], Month=Month_, Year=Year_))
            
        if detectG == 1:
            all_reports.extend(run_notebooks("03-Detection-G", specific_files=["ตรวจราคาG.ipynb"], Month=Month_, Year=Year_, session='g', User=User_))


    # --- Section2.2: Supply Analysis (Session U) ---
    if sessions == "u":
        print("\n📦 2 Data Analysis: คำนวณชุดนอกเขต (U)")
        if web_cpiU == 1:
            all_reports.extend(run_notebooks("01-database", specific_files=["Data_CPI_U.ipynb"], Month=Month_, Year=Year_))

    # --- Section3: Final Output & Upload ---
    if upload == 1:
        print("\n📤 3 Final Output: กำลังส่งออกข้อมูลและอัปโหลด")
        # all_reports.extend(run_notebooks("Final_output", specific_files=["CallOutput.ipynb"], Month=Month_, Year=Year_, session='g'))



📊 2 Data Analysis: คำนวณชุดทั่วไป (G)


  0%|          | [00:00]

In [49]:
print("\n" + "="*85)
print("📊 ตารางสรุปผลการรัน Notebooks ทั้งหมด")
print("="*85)

df = pd.DataFrame(all_reports)
df.index = df.index + 1
df.index.name = 'ลำดับ'

total_files = len(df)
success_count = len(df[df['สถานะ'] == "✅"])
failure_count = total_files - success_count

success_rate = (success_count / total_files) * 100
total_duration = time.time() - start_all

display(df[["ชื่อไฟล์", "ตัวแปรที่ใช้", "เวลา (วินาที)", "รันไปได้ (%)", "สถานะ"]])

print("-" * 85)
print(f"⏱️ เวลารวมทั้งสิ้น: {total_duration:.2f} วินาที")
print(f"✅ สำเร็จ: {success_count}/{total_files} ({success_rate:.1f}%) | ❌ ล้มเหลว: {failure_count} ไฟล์")
print("="*85)


📊 ตารางสรุปผลการรัน Notebooks ทั้งหมด


,ชื่อไฟล์,ตัวแปรที่ใช้,เวลา (วินาที),รันไปได้ (%),สถานะ
ลำดับ,,,,,
1,ShopBKK,-,17.1,100,✅


-------------------------------------------------------------------------------------
⏱️ เวลารวมทั้งสิ้น: 17.12 วินาที
✅ สำเร็จ: 1/1 (100.0%) | ❌ ล้มเหลว: 0 ไฟล์
